In [ ]:
!pip install diffrax equinox optax

In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap, lax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from tqdm.auto import tqdm
import optax

jax.config.update("jax_enable_x64", True)

# ---------- Colab-friendly import of shared Dynamics/Training modules ----------
import os, sys

REPO_URL = "https://github.com/YanchengDu/LiquidCHL.git"
REPO_DIR = "LiquidCHL"

if os.path.isdir("Dynamics") and os.path.isdir("Training"):
    repo_root = "."
elif os.path.isdir(os.path.join("..", "Dynamics")):
    repo_root = ".."
else:
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    repo_root = REPO_DIR

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from Dynamics.Model_A import forward_sim_x_ssolvent_clamp, x_to_phi
from Training.contrastive_hebbian_learning_classifier import (
    daydreaming_contrastive_hebbian_learning_lagrange_JAX_fast_sample_adamw,
)


In [ ]:
# ---------- Shared config: dynamics (training + testing) and training hyperparameters ----------
# Use these in one place so training and testing stay consistent.
from random import randint

DT_DYNAMICS = 1e-1
MAX_STEPS = 30_000
T_END = 300.0  # Integration end time (used by training and testing)

# Training hyperparameters (used by CHL_training_hidden / daydreaming)
GAMMA_LEARNING = 0.02
N_SAMPLE = 300
P_BOUNDARY = 0.3
WIDTH = 0.05
# Seed for training (set to an int for reproducibility, or None for random)
TRAINING_SEED = 859  # randint(0, 1000)
# Model / run layout (used by training cells and parameter sweep)
N_SPECIES = 9
CLAMPED = 4


In [ ]:
# Check if JAX sees GPU
print("JAX version:", jax.__version__)
print("Default backend:", jax.default_backend())
print("Visible devices:", jax.devices())
for i, d in enumerate(jax.devices()):
    print(f"  [{i}] {d.device_kind} (id={d.id})")
x = jnp.ones((3, 3))
print("\nSample array device:", x.devices())
print("GPU in use:", jax.default_backend() == \"gpu\")


In [ ]:

def sample_training_memories(key_seed, sample_size, N, p_boundary, width):
    """Generate one batch of training memories (same logic as daydreaming). Returns (n_sample, N) array."""
    key = jax.random.PRNGKey(key_seed)
    keys = jax.random.split(key, sample_size + 1)
    mem_keys = keys[1:]

    def make_one(k):
        k1, k2, k3, k4 = jax.random.split(k, 4)
        m1 = jax.random.uniform(k3, ()) < 1.0
        m2 = jax.random.uniform(k4, ()) < 0.5
        # sample uniform
        inp0_u = jax.random.uniform(k1, shape=(), minval=0.0, maxval=0.25)
        inp1_u = jax.random.uniform(k2, shape=(), minval=0.0, maxval=0.25)
        # sample near boundary
        inp0_b1 = jax.random.uniform(k1, shape=(), minval=0.125-width, maxval=0.125+width)
        inp1_b1 = jax.random.uniform(k2, shape=(), minval=0.125-width, maxval=0.25)

        inp0_b = jnp.where(m2, inp0_b1, inp1_b1)
        inp1_b = jnp.where(m2, inp1_b1, inp0_b1)

        inp0 = jnp.where(m1, inp0_u, inp0_b)
        inp1 = jnp.where(m1, inp1_u, inp1_b)

        cond = inp0 > inp1
        out = jnp.zeros(N)
        # Conditional assignment using jnp.where
        out = out.at[0].set(inp0).at[1].set(inp1)
        hi, lo = 1.1 / N, 0.25 / N
        val2 = jnp.where(cond, hi, lo)
        val3 = jnp.where(cond, lo, hi)
        out = out.at[2].set(val2).at[3].set(val3)
        sum_fixed = jnp.sum(out[:4])
        sum_rem = jnp.sum(out[4:])
        out = out.at[4:].set(out[4:] / sum_rem * (1 - sum_fixed))
        return out

    mems = jax.vmap(make_one)(mem_keys)
    return np.asarray(mems)


def plot_sampled_memories(mems, title='Sampled training memories (input space)'):
    """Scatter: (mems[:,0], mems[:,1]) colored by above/below sine boundary."""
    mems = np.asarray(mems)
    x, y = mems[:, 0], mems[:, 1]
    above = (x > y).astype(int)
    plt.figure(figsize=(7, 6))
    plt.scatter(x, y, c=above, cmap='coolwarm', s=15, alpha=0.8)
    plt.colorbar(label='Above boundary (1) / Below (0)')
    plt.xlabel('phi₀')
    plt.ylabel('phi₁')
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
def evaluate_and_plot_training(
    chi_learned, miu_learned,
    forward_sim_x_ssolvent_clamp,
    energy_diff_hist=None,
    true_error_hist=None,
    n_species=17,
    clamped=2,
    n_points=1000,
    dt_dynamics=None,
    max_steps=None,
    t_end=None,
    key_seed=0,
    use_log=False,
    vmin=0.0,
    vmax=0.25,
    plot=True,
):
    """
    Single sampling run: (1) training summary, (2) test accuracy vs sine boundary,
    (3) scatter plots (Output1, Output2, Hidden...). Reuses the same n_points for
    evaluation and plotting to avoid running the forward sim twice.
    Set plot=False to return metrics only (for parameter sweeps).
    """
    if dt_dynamics is None: dt_dynamics = DT_DYNAMICS
    if max_steps is None: max_steps = MAX_STEPS
    if t_end is None: t_end = T_END
    N = n_species
    key = jax.random.PRNGKey(key_seed)
    key_x, key_y, key_rand = jax.random.split(key, 3)

    x = jax.random.uniform(key_x, shape=(n_points,), minval=1e-3, maxval=0.25)
    y = jax.random.uniform(key_y, shape=(n_points,), minval=1e-3, maxval=0.25)
    rand_tail = jax.random.normal(key_rand, shape=(n_points, N)) * 0.1 / N + 1 / N

    def make_input(xi, yi, rand_row):
        phi = rand_row.at[:2].set(jnp.array([xi, yi]))
        phi = jnp.clip(phi, 1e-8, 1.0)
        sum_fixed = jnp.sum(phi[:2])
        sum_rem = jnp.sum(phi[2:])
        phi = phi.at[2:].set(phi[2:] / sum_rem * (1 - sum_fixed))
        return phi

    inputs = jax.vmap(make_input)(x, y, rand_tail)

    simulate_batched = jax.vmap(
        lambda phi0: x_to_phi(forward_sim_x_ssolvent_clamp(
            phi0=phi0, chi=chi_learned, mu=miu_learned, clamp=clamped,
            t_end=t_end, dt=dt_dynamics, samples=10, max_steps=max_steps,
        ).ys[-1])
    )
    outputs = simulate_batched(inputs)
    if hasattr(outputs, "block_until_ready"):
        outputs.block_until_ready()

    outputs_np = np.array(outputs)
    x_np = np.array(x)
    y_np = np.array(y)

    # ----- 1. Training curve summary -----
    if plot and energy_diff_hist is not None and true_error_hist is not None:
        energy_hist = np.array(energy_diff_hist)
        error_hist = np.array(true_error_hist)
        print("=== Training summary ===")
        print(f"  Energy diff (F+ - F-):  initial = {energy_hist[0]:.6f},  final = {energy_hist[-1]:.6f}")
        print(f"  True error (swap loss):  initial = {error_hist[0]:.6f},  final = {error_hist[-1]:.6f}")
        if len(energy_hist) > 1:
            print(f"  Energy trend: {'↓' if energy_hist[-1] < energy_hist[0] else '↑'}  (min = {energy_hist.min():.6f})")
        if len(error_hist) > 1:
            print(f"  Error trend:  {'↓' if error_hist[-1] < error_hist[0] else '↑'}  (min = {error_hist.min():.6f})")
        print()

    # ----- 2. Test accuracy (same samples as plots) -----
    true_above = (x_np > y_np).astype(int)
    pred_above = (outputs_np[:, 2] > outputs_np[:, 3]).astype(int)
    accuracy = np.mean(pred_above == true_above)
    n_above, n_below = true_above.sum(), len(true_above) - true_above.sum()
    correct_above = np.sum((pred_above == 1) & (true_above == 1))
    correct_below = np.sum((pred_above == 0) & (true_above == 0))

    if plot:
        print("=== Test accuracy (above vs below sine boundary) ===")
        print(f"  Overall accuracy: {accuracy*100:.2f}%  ({int(np.round(accuracy*n_points))}/{n_points} correct)")
        if n_above > 0:
            print(f"  Above boundary:   {correct_above}/{n_above} correct  ({100*correct_above/n_above:.1f}%)")
        if n_below > 0:
            print(f"  Below boundary:   {correct_below}/{n_below} correct  ({100*correct_below/n_below:.1f}%)")
        print()

    # ----- 3. Scatter plots (same outputs_np) -----
    record_list = [outputs_np[:, i] for i in range(2, n_species - 1)]
    if plot:
        titlelist = ["Output1", "Output2"] + [f"Hidden{i}" for i in range(1, n_species - 4)]
        n_plots = len(record_list)
        n_rows = max(1, int((n_species - 3) / 2))
        fig, axes = plt.subplots(n_rows, 2, figsize=(10, 4 * n_rows), constrained_layout=True)
        axes = np.atleast_1d(axes).flatten()
        for i, ax in enumerate(axes):
            if i >= n_plots:
                ax.set_visible(False)
                continue
            data = record_list[i]
            norm = colors.LogNorm(vmin=max(1e-6, data.min()), vmax=max(1e-6, data.max())) if use_log else colors.Normalize(vmin=vmin, vmax=vmax)
            im = ax.scatter(x_np, y_np, c=data, cmap='coolwarm', s=20, linewidths=0, vmax=0.14)
            fig.colorbar(im, ax=ax).set_label(titlelist[i])
            ax.set_xlabel("phi₀")
            ax.set_ylabel("phi₁")
            ax.set_title(titlelist[i])
            ax.grid(False)
        plt.show()

    eval_metrics = {
        "accuracy": float(accuracy),
        "n_test": n_points,
        "correct_above": int(correct_above),
        "correct_below": int(correct_below),
        "n_above": int(n_above),
        "n_below": int(n_below),
    }
    results = {"x": x_np, "y": y_np, "output1": record_list[0], "output2": record_list[1], "hidden1": record_list[2], "hidden2": record_list[3]}
    return eval_metrics, results, outputs_np

In [ ]:
def CHL_training_hidden(target_memories, n_species, n_epochs, clamped=None, train_more=False,
                  gamma_learning=None, n_sample=None, p_boundary=None, width=None,
                  dt_dynamics=None, max_steps=None, t_end=None, verbose=True, seed=None):
    global chi_learned
    global miu_learned
    # Use config defaults when not provided
    if gamma_learning is None: gamma_learning = GAMMA_LEARNING
    if n_sample is None: n_sample = N_SAMPLE
    if p_boundary is None: p_boundary = P_BOUNDARY
    if width is None: width = WIDTH
    if dt_dynamics is None: dt_dynamics = DT_DYNAMICS
    if max_steps is None: max_steps = MAX_STEPS
    if t_end is None: t_end = T_END

    if seed is None: seed = np.random.randint(0, 1000)
    if verbose: print("seed:", seed)

    rng = np.random.default_rng(seed)

    chi_initial = rng.uniform(low=-15.0, high=15.0, size=(n_species, n_species))
    chi_initial[:4, :4] = rng.normal(loc=0.0, size=(4, 4)) * 0.1
    chi_initial = (chi_initial + chi_initial.T) / 2
    np.fill_diagonal(chi_initial, 0.0)
    chi_initial[-1,:] = 0.0
    chi_initial[:,-1] = 0.0
    chi_initial[0,1] = 0.0
    chi_initial[1,0] = 0.0
    if train_more:
        chi_initial = chi_learned

    if verbose:
        plt.figure(figsize=(8, 5))
        plt.imshow(chi_initial, cmap='coolwarm', interpolation='nearest', vmin=-np.max(np.abs(chi_initial)), vmax=np.max(np.abs(chi_initial)))
        plt.colorbar()
        plt.title('Initial Interaction Matrix')
        plt.show()

    miu_initial = rng.normal(loc=0.0, size=(n_species)) * 0.0
    miu_initial[0] = miu_initial[1] = miu_initial[-1] = 0.0
    if train_more:
        miu_initial = miu_learned

    chi_learned, miu_learned, energy_diff_hist, true_error_hist = daydreaming_contrastive_hebbian_learning_lagrange_JAX_fast_sample_adamw(
        target_memories=target_memories,
        chi_initial=chi_initial,
        miu_initial=miu_initial,
        V=1.0,
        dt_dynamics=dt_dynamics,
        n_steps_dynamics=max_steps,
        gamma_learning=gamma_learning,
        n_epochs=n_epochs,
        n_sample=n_sample,
        sample_random_each_epoch=False,
        p_boundary=p_boundary,
        width=width,
        t_end=t_end,
        clamped=clamped,
        print_energy=False,
        key_seed=seed,
    )

    if verbose:
        plt.figure(figsize=(8, 5))
        plt.imshow(chi_learned, cmap='coolwarm', interpolation='nearest', vmin=-np.max(np.abs(chi_learned)), vmax=np.max(np.abs(chi_learned)))
        plt.colorbar()
        plt.title('Learned Interaction Matrix')
        plt.show()
        chi_change = chi_learned - chi_initial
        plt.figure(figsize=(8, 5))
        plt.imshow(chi_change, cmap='coolwarm', interpolation='nearest',vmin=-np.max(np.abs(chi_change)), vmax=np.max(np.abs(chi_change)))
        plt.colorbar()
        plt.title('Change in Interaction Matrix')
        plt.show()

    def plot_grayscale_blocks(values, orientation='horizontal', cmap='gray', figsize=(8,1), vmin=None, vmax=None, show_colorbar=False):
        """
        Plot a 1D list/array `values` as adjacent grayscale blocks.
        orientation: 'horizontal' (1 x n) or 'vertical' (n x 1)
        """
        arr = np.asarray(values, dtype=float)
        if vmin is None: vmin = np.nanmin(arr)
        if vmax is None: vmax = np.nanmax(arr)
        if orientation == 'horizontal':
            im = arr.reshape(1, -1)
            fig, ax = plt.subplots(figsize=figsize)
            mappable = ax.imshow(im, cmap=cmap, aspect='auto', interpolation='nearest', vmin=vmin, vmax=vmax)
            ax.set_yticks([]); ax.set_xticks([])
        else:
            im = arr.reshape(-1, 1)
            fig, ax = plt.subplots(figsize=(figsize[1], figsize[0]) if isinstance(figsize, tuple) else (1, len(arr)/4))
            mappable = ax.imshow(im, cmap=cmap, aspect='auto', interpolation='nearest', vmin=vmin, vmax=vmax)
            ax.set_xticks([]); ax.set_yticks([])

        if show_colorbar:
            plt.colorbar(mappable=mappable, ax=ax, orientation='horizontal' if orientation=='horizontal' else 'vertical')
        plt.tight_layout()
        plt.show()

    if verbose:
        plot_grayscale_blocks(miu_learned, orientation='horizontal', figsize=(chi_learned.shape[0], 1.0), show_colorbar=True)
        plt.figure(figsize=(8, 5))
        plt.plot(energy_diff_hist)
        plt.xlabel('Epoch')
        plt.ylabel('Average Energy Difference (F+ - F-)')
        plt.title('Contrastive Hebbian Learning Progress')
        plt.grid(True)
        plt.show()
        plt.figure(figsize=(8, 5))
        plt.plot(true_error_hist)
        plt.xlabel('Epoch')
        plt.ylabel('Average True Error')
        plt.title('Contrastive Hebbian Learning Progress')
        plt.grid(True)
        plt.show()
        print("chi_learned:", chi_learned)
        print("miu_learned:", miu_learned)
    return chi_learned, miu_learned, energy_diff_hist, true_error_hist

In [ ]:
# Generate and plot one batch of training memories (same as first epoch of training)
mems = sample_training_memories(
    key_seed=TRAINING_SEED,
    sample_size=N_SAMPLE,
    N=N_SPECIES,
    p_boundary=P_BOUNDARY,
    width=WIDTH,
)
plot_sampled_memories(mems, title='My sampled memories')

In [ ]:
chi_learned, miu_learned, energy_diff_hist, true_error_hist = CHL_training_hidden(
    target_memories=None, n_species=N_SPECIES, n_epochs=500, clamped=CLAMPED, train_more=False, 
    seed=TRAINING_SEED, 
)

# Single sampling: evaluation + scatter plots (saves running forward sim twice)
eval_metrics, results, outputs_np = evaluate_and_plot_training(
    chi_learned, miu_learned,
    forward_sim_x_ssolvent_clamp,
    energy_diff_hist=energy_diff_hist,
    true_error_hist=true_error_hist,
    n_species=N_SPECIES, clamped=2, n_points=1000, key_seed=TRAINING_SEED if TRAINING_SEED is not None else 0,
    use_log=False,
)

In [ ]:
# Single sampling: evaluation + scatter plots (saves running forward sim twice)
eval_metrics, results, outputs_np = evaluate_and_plot_training(
    chi_learned, miu_learned,
    forward_sim_x_ssolvent_clamp,
    energy_diff_hist=energy_diff_hist,
    true_error_hist=true_error_hist,
    n_species=N_SPECIES, clamped=2, n_points=1000, key_seed=TRAINING_SEED if TRAINING_SEED is not None else 0,
    use_log=False,
)